# 02 · Continual learning with expanding polynomial features

**Main question.** Can incremental polynomial expansion learn new information while retaining what was learned before?

The central idea: expand the polynomial feature dimensions as new experiences arrive, then control how the existing and the newly introduced dimensions learn, to balance plasticity, accuracy and forgetting.

* **Part I: class-incremental learning (CIL)**, the primary research setting. Ten classes arrive in five experiences of two classes each, one softmax head, Split MNIST as the real-data benchmark.
* **Part II: sequential *binary* learning under spatial distribution shift.** Two classes throughout; only *where* the inputs come from changes. This is **not** CIL and is kept separate.

`01_joint_training.ipynb` gives the non-continual reference. Every cell below calls package code; no result files are read, and the experiments are run here (reduced where the full study is expensive; the full per-degree study is `experiments/per_degree_lr.py`).

| Section | Question | RQ |
|---|---|---|
| I.1 | What is the CIL protocol? | |
| I.2 | Does zero-initialised expansion preserve predictions? | RQ2 |
| I.3 | Does expansion provide useful capacity? | RQ2 |
| I.4 | How do sampling and replay affect learning? | RQ3 |
| I.5 | How should old and new dimensions be updated? | RQ4 |
| I.6 | Does per-degree learning-rate control help? | RQ5 |
| I.7 | Are the results stable on Split MNIST? | RQ6 |
| I.8 | Research summary | all |

## Setup and configuration

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = next(
    r for r in (Path.cwd(), Path.cwd().parent) if (r / "src" / "polynomial_cl").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))

from polynomial_cl import metrics
from polynomial_cl.continual import (
    CONDITIONS,
    STRATEGIES,
    TASK_ORDER,
    as_degree_rates,
    candidate_vectors,
    degree_group_rows,
    degree_of_feature,
    holdout_split,
    make_feature_data,
    make_task_plan,
    old_class_rows,
    per_seed_rows,
    run_comparison,
    run_condition,
    run_per_degree_study,
    run_sampling_comparison,
    run_strategy_comparison,
    sampling_rows,
    strategy_diagnostic_rows,
    summary_rows,
)
from polynomial_cl.continual.degree_rates import group_sizes
from polynomial_cl.continual.degree_study import study_pairs, study_rows
from polynomial_cl.data import load_projected_mnist, make_synthetic_2d
from polynomial_cl.plotting import continual as P

plt.rcParams.update({"figure.dpi": 80})

SEEDS = [0, 1, 2, 3, 4]
CIL = {"epochs": 3, "learning_rate": 0.1, "batch_size": 128, "memory_per_class": 200}
# Mini-batch policy, stated explicitly: the cells below that do not compare policies keep the
# protocol of the earlier experiments, "shuffled". The library default is now "class_balanced"
# (see I.4); I.4 and I.6 compare both policies explicitly.
CIL["sampling"] = "shuffled"

# Part I · Class-incremental learning (CIL)

## I.1 · CIL protocol

**Question.** What exactly does the model see, in which order, and what is it evaluated on?

**Why it matters.** Forgetting and capacity claims are only meaningful if the protocol is fixed and leak-free.

**Protocol (unchanged from the existing experiments).**

* **Data.** Split MNIST: images are normalised and mapped to **16 dimensions by a fixed random orthonormal projection** (never fitted to data). A synthetic ten-class ring is used for a quick protocol check.
* **Experiences.** Five experiences of **two new classes each**, in the fixed order (0,1), (2,3), (4,5), (6,7), (8,9).
* **Model.** One softmax head over ten output columns. Columns of classes that have **not been introduced yet are never trained and are excluded from prediction**, so future classes cannot influence anything before their experience.
* **Features.** Degree-scaled polynomial monomials of the 16 inputs; the *expanding* condition uses degrees 1, 2, 3, 3, 3 over the five experiences, new coefficients start at zero.
* **Replay.** Up to 200 stored examples per old class, mixed with the current experience; strategies differ in what may update (I.5).
* **Training.** 3 epochs per experience, learning rate 0.1, batch size 128. **Sampling: `shuffled`, set explicitly in the configuration cell** (every example of the current experience and replay is read once per epoch). This is the protocol of the earlier experiments; the library default for new runs is `class_balanced`, and I.4 and I.6 compare both policies explicitly.
* **Evaluation.** After each experience, accuracy on the test inputs of every experience seen so far (the **accuracy matrix**); forgetting of an experience is its best earlier accuracy minus its accuracy at the end, averaged over the first four experiences (the final experience is excluded, as it has had no chance to be forgotten). Rates and strategies are never chosen on the test set.
* **Seeds.** 0–4 (data order, replay selection and shuffling depend on the seed); results are mean ± sample standard deviation (ddof = 1).

The cell below runs the quick protocol check on the synthetic ring (all conditions, then all six strategies), shows the training data by experience, and loads Split MNIST if the files are available.

In [ ]:
x_tr, y_tr, x_te, y_te, _ = make_synthetic_2d(0)
fig, ax = plt.subplots(figsize=(4.6, 4.6))
for task, classes in enumerate(TASK_ORDER):
    mask = np.isin(y_tr, classes)
    ax.scatter(
        x_tr[mask, 0],
        x_tr[mask, 1],
        s=8,
        alpha=0.6,
        label=f"experience {task + 1}: classes {classes}",
    )
ax.set_aspect("equal")
ax.legend(fontsize=7)
ax.set_title("Synthetic ring: training data by experience")
plt.show()

ring = make_feature_data(x_tr, y_tr, x_te, y_te)
ring_conditions = run_comparison(ring, SEEDS, **CIL)
print("synthetic ring, expansion conditions")
print(metrics.format_table(summary_rows(ring_conditions)))
ring_strategies = run_strategy_comparison(ring, SEEDS, **CIL)
print("\nsynthetic ring, update strategies")
print(
    metrics.format_table(
        summary_rows(
            ring_strategies,
            key="strategy",
            metric_names=(
                "final_mean_accuracy",
                "final_current_accuracy",
                "mean_forgetting",
            ),
        )
    )
)

b_run = next(
    r
    for r in ring_strategies["runs"]
    if r["strategy"] == "strategy_b" and r["seed"] == 0
)
print(
    "\nStrategy B on the ring (seed 0): trainable coefficients",
    b_run["n_trainable_by_task"],
    "| null-space dimension",
    b_run["null_dim_by_task"],
)

run = ring_conditions["runs"][0]
unseen = np.isnan(
    np.array(
        [
            [np.nan if v is None else v for v in row]
            for row in run["accuracy_by_class_matrix"]
        ]
    )
)
print(
    "\nper-class accuracy is only defined for classes already introduced (seed 0, 'x' = never evaluated):"
)
for t, row in enumerate(unseen):
    print(f"  after experience {t + 1}:", " ".join("x" if u else "o" for u in row))

In [ ]:
MNIST_DIR = ROOT / "data" / "mnist"
try:
    HAVE_MNIST = True
    mnist = make_feature_data(
        *load_projected_mnist(MNIST_DIR, projection_seed=0, download=True)
    )
    print(
        f"Split MNIST: {len(mnist.train_y)} training / {len(mnist.test_y)} test images, 16 projected dimensions"
    )
except OSError as error:  # no local files and no network
    HAVE_MNIST = False
    print(
        "Split MNIST unavailable; the Split MNIST cells are skipped:",
        type(error).__name__,
    )

**Observed (synthetic ring, seeds 0–4).** The three expansion conditions end at 0.709 (`fixed_linear`), 0.909 (`expanding`) and 0.908 (`fixed_capacity`) final mean accuracy; `fixed_linear` reaches 0.000 on the newest experience (a linear head cannot separate the ring classes). Without replay the final mean accuracy is 0.455 and forgetting 0.621. `strategy_b` and `strategy_b_new` end at 0.189 with 0.000 newest accuracy and 0.000 forgetting: on the ring Strategy B has 6 trainable coefficients in the first experience and **0 from the second on, because the null space of the reference design is empty** (null-space dimension 0). Per-class accuracy is defined only for classes already introduced (the printed `o`/`x` pattern is lower-triangular).

**Interpretation.** The protocol behaves as specified: classes appear in order and only introduced classes are evaluated. The ring is a two-dimensional check of the protocol and of the code paths; its Strategy B result says that a 3–10 feature model leaves no update freedom under that constraint, and it is not a statement about real data.

## I.2 · Expansion and preservation

**Question.** Does zero-initialised polynomial expansion preserve previous predictions?

**Why it matters.** Expansion adds coefficients. If it changed the function at the moment of expansion, every later change in accuracy would be confounded with the expansion itself.

**Experiment.** (a) *Binary, synthetic data:* train a degree-2 model on the sinusoidal target, record its scores on a fixed set of 4000 probe inputs (half inside the domain, half in $[-3,3]^2$), expand to degree 4 with all new coefficients set to zero, record the scores again, and compare. Then train the expanded model further and compare again. (b) *Ten-class softmax head:* the `expanding` condition on the synthetic ring records the largest change in any logit at every expansion, before any optimisation step.

**Configuration.** (a) Learning rate 0.5, 2000 epochs, then 500 more epochs of further training; seed 0. (b) The CIL protocol of I.1.

In [ ]:
from polynomial_cl.binary import combine_splits, make_experiences
from polynomial_cl.models.binary import PolynomialLogisticModel
from polynomial_cl.plotting import synthetic as S
from polynomial_cl.training.binary import train

exps = make_experiences(seed=0, n_train=200, n_test=500)
x, y, _ = combine_splits([e.train for e in exps])
small = train(PolynomialLogisticModel(2), x, y, lr=0.5, epochs=2000, seed=0).model
wide = small.expanded(4)  # new coefficients are zero
further = train(wide, x, y, lr=0.5, epochs=500, seed=1).model

rng = np.random.default_rng(0)
probe = np.vstack([rng.uniform(-1, 1, (2000, 2)), rng.uniform(-3, 3, (2000, 2))])
print(
    f"polynomial dimensions: {small.n_features} -> {wide.n_features}; "
    f"new coefficients all zero: {not wide.w[small.n_features :].any()}"
)
inside = np.all(np.abs(probe) <= 1, axis=1)
for name, m in (
    ("immediately after expansion", wide),
    ("after 500 further epochs", further),
):
    diff = np.abs(m.score(probe) - small.score(probe))
    flips = m.predict(probe) != small.predict(probe)
    print(
        f"{name}: inside the data domain max |score diff| = {diff[inside].max():.2e}, "
        f"flipped {flips[inside].mean():.4f} | outside: max |score diff| = {diff[~inside].max():.2e}, "
        f"flipped {flips[~inside].mean():.4f}"
    )

fig, axs = plt.subplots(1, 4, figsize=(17, 4))
S.plot_predictions(
    small, axs[0], samples=(x, y), title=f"degree {small.degree}: trained"
)
S.plot_predictions(
    wide, axs[1], samples=(x, y), title=f"degree {wide.degree}: expanded, not trained"
)
S.plot_prediction_changes(small, wide, axs[2], title="changed predictions at expansion")
S.plot_predictions(
    further,
    axs[3],
    samples=(x, y),
    title=f"degree {further.degree}: after further training",
)
plt.tight_layout()
plt.show()

run = run_condition(
    "expanding",
    ring,
    *make_task_plan(ring.train_y, 0, 200),
    seed=0,
    epochs=3,
    sampling=CIL["sampling"],
)
print(
    "\nten-class head, synthetic ring: features per experience",
    run["feature_count_by_task"],
)
print(
    "max |logit change| at each experience's expansion (before any update):",
    [f"{v:.1e}" for v in run["expansion_max_logit_changes"]],
)

**Observed.** (a) The degree-2 binary model (6 coefficients) was expanded to degree 4 (15 coefficients); all 9 new coefficients are exactly zero. Immediately after expansion the largest score difference on the probe set is $1.8\times10^{-15}$ inside the data domain and $7.1\times10^{-15}$ outside it, and no prediction flips; the change map is empty (the first two panels are identical). After 500 further epochs of training 2.2% of the in-domain probe predictions flip (largest score difference 2.4) and 25.5% of the outside-domain predictions flip (largest difference 106). (b) For the ten-class head on the ring the feature count grows 3 → 6 → 10 and the largest logit change at the two expansions is exactly 0.

**Interpretation.** Zero-initialised expansion preserves the function to rounding error at the moment of expansion, for scores as well as predictions; it adds capacity without changing any output. That is a property of the expansion step only. Once training resumes the predictions change, mostly where the data give no constraint (outside the domain); whether that change is forgetting is measured by the accuracy matrices below.

**Caveat.** One model of each kind and one probe set are shown; the unit tests check the invariant for more degrees, seeds and for the ten-class head.

## I.3 · Does expansion provide useful capacity?

**Question.** Does adding polynomial dimensions improve learning compared with keeping the capacity fixed?

**Why it matters.** More dimensions are not automatically useful: a larger model can be harder to train, or can spend its new capacity overwriting old behaviour.

**Experiment.** Three existing conditions on Split MNIST, identical in everything but the degree schedule: `fixed_linear` (degree 1 throughout, 17 features), `expanding` (degrees 1, 2, 3, 3, 3), `fixed_capacity` (degree 3 from the first experience, 969 features). Same seeds, replay, epochs, learning rate and batch size (see I.1).

**Configuration.** Seeds 0–4, 3 epochs per experience, learning rate 0.1 for all conditions.

In [ ]:
if HAVE_MNIST:
    capacity = run_comparison(mnist, SEEDS, **CIL)
    by_condition = {
        c: [r for r in capacity["runs"] if r["condition"] == c] for c in CONDITIONS
    }
    print(metrics.format_table(summary_rows(capacity)))
    print("\nfeatures per experience (seed 0):")
    print(
        metrics.format_table(
            [
                {
                    "condition": c,
                    **{
                        f"exp {t + 1}": n
                        for t, n in enumerate(runs[0]["feature_count_by_task"])
                    },
                }
                for c, runs in by_condition.items()
            ]
        )
    )
    fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
    P.plot_cil_accuracy_over_experiences(
        by_condition, ax=axs[0], title="accuracy over experiences"
    )
    P.plot_cil_forgetting(by_condition, ax=axs[1], title="mean forgetting")
    plt.tight_layout()
    plt.show()
    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    for ax, (c, runs) in zip(axs, by_condition.items()):
        P.plot_cil_accuracy_matrix(runs, ax=ax, title=f"{c}: seed-mean accuracy matrix")
    plt.tight_layout()
    plt.show()

**Observed (Split MNIST, default rate 0.1, 3 epochs, seeds 0–4; mean ± sample std).**

| condition | features | final mean acc. | newest-experience acc. | forgetting |
|---|---|---|---|---|
| `fixed_linear` | 17 | 0.176 ± 0.000 | 0.877 | 0.926 ± 0.001 |
| `expanding` | 17 → 153 → 969 | 0.246 ± 0.002 | 0.902 | 0.858 ± 0.002 |
| `fixed_capacity` | 969 | 0.256 ± 0.001 | 0.902 | 0.846 ± 0.002 |

**Interpretation.** Expansion helps relative to a fixed linear model: +0.070 final accuracy and −0.068 forgetting. The newest-experience accuracy is almost the same (0.877 vs 0.902), so the gain is in what is *retained*, not in learning the newest classes. `expanding` and `fixed_capacity` end with the same dimensions and similar results; the 0.010 difference exceeds the seed spread, but it does not show that growing gradually is better, only that capacity matters. More dimensions are not by themselves useful information: all three conditions lose most old classes (forgetting above 0.84).

**Caveats.** These runs use the default rate and 3 epochs, far from convergence, and the README documents that conclusions at default rates can change once rates are tuned per condition; they compare mechanisms, not a ranking. Seeds vary shuffling and replay selection but not the projection or the data, so the standard deviations understate the uncertainty.

## I.4 · Sampling and replay

**Question.** How do sample order, class balance and replay affect class-incremental learning?

**Why it matters.** With replay, the training stream of experience *t* contains few examples of the new classes next to many stored old ones. Plain shuffling therefore decides how much each class is seen, and that alone can move accuracy and forgetting.

**Experiment.** The two existing sampling policies, `shuffled` (the default: each example once per epoch, in random order) and `class_balanced` (each batch gives every class with examples an equal share of slots; a class with fewer examples than slots is repeated), crossed with all six strategies, with identical seeds, epochs, learning rate, batch size and replay memory. Replay is isolated by comparing `ordinary` (replay) with `ordinary_no_replay`. For a fair budget the tables report the **number of updates and of examples processed** and how often a stored example was used.

**Configuration.** Seeds 0–4, 3 epochs, learning rate 0.1, batch size 128, up to 200 stored examples per class.

In [ ]:
if HAVE_MNIST:
    sampling = run_sampling_comparison(
        mnist,
        SEEDS,
        learning_rate=CIL["learning_rate"],
        epochs=CIL["epochs"],
        batch_size=CIL["batch_size"],
        memory_per_class=CIL["memory_per_class"],
    )
    focus = ("ordinary", "ordinary_no_replay")
    columns = (
        "final_mean_accuracy",
        "final_current_accuracy",
        "mean_forgetting",
        "mean_old_takeover_fraction",
    )
    for policy, result in sampling.items():
        print(f"== {policy}")
        print(
            metrics.format_table(
                summary_rows(
                    {"summary": {s: result["summary"][s] for s in focus}},
                    key="strategy",
                    metric_names=columns,
                )
            )
        )
    print(
        "\nexamples processed per run (summed over the five experiences, mean over seeds):"
    )
    print(
        metrics.format_table(
            [r for r in sampling_rows(sampling) if r["strategy"] in focus]
        )
    )

    fig, axs = plt.subplots(1, 3, figsize=(19, 4.4))
    P.plot_cil_tradeoff(
        [
            {
                "label": f"{s} ({pol[:4]})",
                "runs": [r for r in res["runs"] if r["strategy"] == s],
                "sampling": pol,
            }
            for pol, res in sampling.items()
            for s in focus
        ],
        ax=axs[0],
    )
    logs = {
        pol: next(
            r for r in res["runs"] if r["strategy"] == "ordinary" and r["seed"] == 0
        )["sampling_by_task"]
        for pol, res in sampling.items()
    }
    P.plot_sampling_counts(
        logs, ax=axs[1], title="`ordinary`, experience 5: examples per class"
    )
    P.plot_cil_per_class_accuracy(
        {
            pol: [r for r in res["runs"] if r["strategy"] == "ordinary"]
            for pol, res in sampling.items()
        },
        ax=axs[2],
        title="`ordinary`: per-class accuracy after experience 5",
    )
    plt.tight_layout()
    plt.show()

**Observed (Split MNIST, seeds 0–4).**

* *Shuffled (default).* `ordinary` 0.246 final accuracy, forgetting 0.858, **96.1% of old-class test inputs taken over by a new class**; without replay 0.192, 0.925 and 99.6%. In the per-class plot the old classes are mostly near 0 and classes 8 and 9 reach 0.87 and 0.93.
* *Class-balanced.* `ordinary` 0.675 final accuracy, forgetting 0.073, takeover 0.071, but only 0.452 on the newest experience (shuffled: 0.902). Without replay, class-balanced equals shuffled in results (0.192, 0.925): there are no old examples to balance.
* *Examples processed.* Both policies perform 1506 updates per run. Shuffled processes 192,000 examples (each stored example 3 times); class-balanced 192,768 (+0.4%), but unevenly: the least-drawn class has each stored example used 0.68 times and the most-drawn 20.2 times (small replay pools are repeated).

**Interpretation.** Replay with 200 stored examples per class changes little when sampling is plain shuffling (+0.054 accuracy), because the stream is dominated by the roughly 6000 new examples per class; how the stream is sampled changes far more (+0.43 accuracy, −0.79 forgetting at equal update count and +0.4% examples). It is a stability–plasticity trade-off, not a free gain: newest-experience accuracy halves. The class-balanced policy also repeats stored examples about 20 times, so its advantage is not only 'more balanced'; the data do not separate the effect of balance from the effect of repetition.

**Caveat.** Default rate and 3 epochs; the per-degree study (I.6) shows that tuned scalar rates move these numbers a lot.

## I.5 · Update strategies

**Question.** How should old and new polynomial dimensions be updated?

**Why it matters.** After expansion, the old coefficients encode what was learned, but they are also the ones whose change causes forgetting; the new coefficients are free capacity. The six existing strategies differ in what is allowed to move.

| strategy | what is allowed to update |
|---|---|
| `ordinary` | everything, with replay (the baseline) |
| `ordinary_no_replay` | everything, **no replay**: isolates the effect of replay |
| `small_old_lr` | everything, but coefficients that already existed (old feature row **and** old class column) take steps scaled by 0.1 |
| `frozen_old` | old coefficients frozen (scale 0); only new rows or new class columns train |
| `strategy_b` | every coefficient, but the update is restricted to the **null space** of the design matrix of a few stored reference inputs of each old class (one shared basis for all output columns): the logits at those inputs then do not change (to first order). Only at those finitely many inputs; nowhere else is anything guaranteed |
| `strategy_b_new` | old coefficients frozen; the new ones move inside a null space with a **separate basis for the old-class columns and for the new-class columns**: the old-class columns can only move through the new feature rows (their old rows are frozen), the new-class columns through every row, each inside the null space of the reference design |

Strategy B needs the allowed update space (the *null dimension*) to be non-empty; a small null space means little can be learned. `n_trainable` counts the coefficients allowed to move at all; `l1_update` is the size of the actual change.

**Experiment.** Compare all six on Split MNIST with the `shuffled` default, using accuracy, forgetting, class-level behaviour, trainable dimensions and coefficient updates. These are the same runs as the `shuffled` half of I.4 (same seeds and protocol); a sweep of the number of reference inputs follows.

**Configuration.** Seeds 0–4; Strategy B uses 3 reference inputs per old class unless stated.

In [ ]:
if HAVE_MNIST:
    strategy = sampling["shuffled"]
    by_strategy = {
        s: [r for r in strategy["runs"] if r["strategy"] == s] for s in STRATEGIES
    }
    print(
        metrics.format_table(
            summary_rows(
                strategy,
                key="strategy",
                metric_names=columns
                + ("mean_old_flip_fraction", "mean_old_logit_change"),
            )
        )
    )

    fig, axs = plt.subplots(2, 3, figsize=(14, 8))
    for ax, (s, runs) in zip(axs.ravel(), by_strategy.items()):
        P.plot_cil_accuracy_matrix(runs, ax=ax, title=s)
    plt.tight_layout()
    plt.show()

    fig, axs = plt.subplots(1, 3, figsize=(20, 4.4))
    P.plot_cil_tradeoff(
        [{"label": s, "runs": runs} for s, runs in by_strategy.items()], ax=axs[0]
    )
    P.plot_cil_takeover_flips(by_strategy, ax=axs[1])
    P.plot_cil_per_class_accuracy(
        by_strategy, ax=axs[2], title="per-class accuracy after experience 5"
    )
    plt.tight_layout()
    plt.show()

    for s in ("ordinary", "frozen_old", "strategy_b", "strategy_b_new"):
        print(
            f"\n== {s} (seed 0): trainable coefficients, null-space sizes, update size"
        )
        print(metrics.format_table(strategy_diagnostic_rows(by_strategy[s][0])))
    print("\n== old-class behaviour of `ordinary` (seed 0)")
    print(metrics.format_table(old_class_rows(by_strategy["ordinary"][0])))

In [ ]:
if HAVE_MNIST:
    rows = []
    for n_ref in (1, 3, 10, 30, 100, 200):
        res = run_strategy_comparison(
            mnist, SEEDS[:3], strategies=("strategy_b",), n_ref_per_class=n_ref, **CIL
        )
        s = res["summary"]["strategy_b"]
        rows.append(
            {
                "refs_per_class": n_ref,
                "final_mean_acc": f"{s['final_mean_accuracy']['mean']:.3f}",
                "final_current_acc": f"{s['final_current_accuracy']['mean']:.3f}",
                "forgetting": f"{s['mean_forgetting']['mean']:.3f}",
                "old_takeover": f"{s['mean_old_takeover_fraction']['mean']:.3f}",
                "null_dim_last_exp": int(
                    np.mean([r["null_dim_by_task"][-1] for r in res["runs"]])
                ),
            }
        )
    print(
        "Strategy B: effect of the number of reference inputs per old class (seeds 0-2)"
    )
    print(metrics.format_table(rows))

**Observed (shuffled sampling, Split MNIST, seeds 0–4).**

| strategy | final acc. | newest acc. | forgetting | old taken over | old flipped among old |
|---|---|---|---|---|---|
| `ordinary` | 0.246 | 0.902 | 0.858 | 0.961 | 0.385 |
| `ordinary_no_replay` | 0.192 | 0.902 | 0.925 | 0.996 | 0.139 |
| `small_old_lr` | 0.260 | 0.897 | 0.838 | 0.940 | 0.078 |
| `frozen_old` | 0.274 | 0.893 | 0.820 | 0.925 | 0.021 |
| `strategy_b` | 0.538 | 0.683 | 0.405 | 0.381 | 0.074 |
| `strategy_b_new` | 0.543 | 0.496 | 0.336 | 0.304 | 0.021 |

Seed standard deviations are ≤ 0.002 except Strategy B (0.009). The restrictions are visible in the diagnostics: `frozen_old` trains 1938 coefficients in experiences 4–5 (vs 7752 and 9690 for `ordinary`); `strategy_b` allows a null space of about 950 dimensions in experiences 3–5; in `strategy_b_new` the old-class columns have **no** free direction from experience 4 on (null dimension 0) while the new-class columns keep about 950. Strategy B's reference inputs act as a dial: with 1, 3, 10, 30, 100, 200 references per old class the final accuracy is 0.426, 0.538, 0.628, 0.599, 0.275, 0.199, the newest accuracy 0.819 → 0.000, the takeover 0.648 → 0.001, and the last null dimension 961 → 0.

**Interpretation.** Strategies that damp old coefficients (`small_old_lr`, `frozen_old`) strongly reduce flips among old classes (0.385 → 0.021) but barely reduce takeover (0.96 → 0.93): the old classes are lost mainly because new-class logits win on old inputs, not because old logits are scrambled. Strategy B reduces takeover (to 0.38 / 0.30) at the price of newest-experience accuracy (0.68 / 0.50), and the reference count moves along that trade-off; with 200 references nothing can be learned. A single ranking by final accuracy would hide this.

**Caveats.** All strategies use the same default rate (0.1) and 3 epochs, and the README documents that Strategy B's large advantage at default rates came from an under-trained shared-rate baseline and does not survive rate tuning; these results do not show that any strategy is better, only how each one behaves.

## I.6 · Per-degree learning rates

**Question.** Does independently controlling the learning rate of each polynomial degree improve the accuracy/forgetting trade-off?

**Why it matters.** Old degrees hold what was learned; newly introduced degrees are free capacity. One scalar rate cannot treat them differently.

**Two distinct learning-rate mechanisms (they cannot be combined).**

* `feature_power` scales the step size of **each individual feature** by its RMS value (`lr * (max_rms / rms_i) ** p`); it does not decide which dimensions exist.
* A **per-degree rate vector** gives one step size to **each polynomial degree group** (degree 0, 1, 2, 3). The vector `[0.1, 0.1, 0, 0.1]` means: degree-2 coefficients get rate 0, i.e. they are never updated, and do not count as trainable. A scalar `0.1` is the same rate for every group.

The degree of a coefficient is the degree of its feature row; *which* features exist is decided by the degree schedule of I.3. The code rejects a configuration that sets both mechanisms.

**Experiment.** (a) Show the groups and the candidate vectors (each multiplies one group of the shared rate by 0.25, 0.5, 2 or 4). (b) Check that the two mechanisms are rejected together and that a zero rate really freezes its group. (c) A **reduced** version of the existing study: on a held-out validation part of the training set (seed 0) it first selects the best **shared scalar rate** from a grid (0.01 … 30), then tries each candidate vector (one group at a time multiplied by 0.25, 0.5, 2 or 4, plus a combined vector) and selects one from the validation Pareto front; the shared and the selected vector are then evaluated on the test set with different seeds (1, 2), for `ordinary` under both sampling policies. The full study (all strategies, 5 selection and 7 test seeds) is `experiments/per_degree_lr.py`; nothing is read from saved results.

**Configuration.** (a)/(b): base rate 0.1. (c): shared rate chosen by the study on validation data (printed in the table), selection seed 0, test seeds 1 and 2, 3 epochs, batch size 128.

In [ ]:
index = degree_of_feature(16, 3)
print("coefficient rows per degree group (degrees 0..3):", group_sizes(index).tolist())
print("rate vector for the scalar 0.1:", as_degree_rates(0.1).tolist())
vectors = candidate_vectors(0.1)
print(f"{len(vectors)} candidate vectors; for example:")
for name in ("shared", "d1x0.25", "d3x4"):
    print(f"  {name:8s} {vectors[name]}")

In [ ]:
if HAVE_MNIST:
    plan = make_task_plan(mnist.train_y, 0, CIL["memory_per_class"])
    settings = {k: CIL[k] for k in ("epochs", "batch_size", "sampling")}
    try:
        run_condition(
            "expanding",
            mnist,
            *plan,
            seed=0,
            learning_rate=[0.1] * 4,
            feature_power=1.0,
            **settings,
        )
    except ValueError as error:
        print("combining both mechanisms is rejected:", error)

    frozen = run_condition(
        "expanding",
        mnist,
        *plan,
        seed=0,
        learning_rate=[0.1, 0.1, 0.0, 0.1],
        **settings,
    )
    scalar = run_condition(
        "expanding", mnist, *plan, seed=0, learning_rate=0.1, **settings
    )
    print(
        "\ntrainable coefficients per experience  scalar rate:",
        scalar["n_trainable_by_task"],
        "| degree-2 rate 0:",
        frozen["n_trainable_by_task"],
    )
    print(
        "\nper degree group, degree-2 rate 0 (the degree-2 rows show the frozen group):"
    )
    print(
        metrics.format_table(
            [r for r in degree_group_rows(frozen) if r["degree"] in (1, 2)]
        )
    )

In [ ]:
if HAVE_MNIST:
    train_x, train_y, _, _ = load_projected_mnist(
        MNIST_DIR, projection_seed=0, download=False
    )
    validation = make_feature_data(
        *holdout_split(train_x, train_y, 0.1)
    )  # held-out part is the evaluation set
    study = run_per_degree_study(
        validation,
        mnist,
        [0],
        [1, 2],
        strategies=("ordinary",),
        epochs=CIL["epochs"],
        batch_size=CIL["batch_size"],
        memory_per_class=CIL["memory_per_class"],
    )
    print(
        "REDUCED RUN: selection seeds",
        study["selection_seeds"],
        "| test seeds",
        study["test_seeds"],
    )
    print(
        "(* = selected on validation data, ^ = on the validation Pareto front of accuracy and forgetting)"
    )
    print(metrics.format_table(study_rows(study)))

    pairs = study_pairs(study)
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    P.plot_cil_tradeoff(
        [{"label": k, "runs": r, "sampling": smp} for k, (r, smp) in pairs.items()],
        ax=axs[0],
        annotate=False,
    )
    P.plot_cil_forgetting({k: r for k, (r, _) in pairs.items()}, ax=axs[1])
    plt.tight_layout()
    plt.show()
    for cell in study["cells"]:
        P.plot_rate_vectors_with_results(
            cell["vectors"],
            cell["runs"],
            cell["selected"],
            title=f"`ordinary`, {cell['sampling']}: rates alongside results",
        )
        plt.tight_layout()
        plt.show()

In [ ]:
if HAVE_MNIST:
    cell = next(c for c in study["cells"] if c["sampling"] == "class_balanced")
    for name in ("shared", cell["selected"]):
        runs = cell["runs"][name]
        fig, axs = plt.subplots(1, 4, figsize=(21, 4))
        for ax, (key, title) in zip(
            axs,
            [
                ("mean_gradient_l1", "raw gradient L1 per update"),
                ("coef_change_l1", "actual coefficient change (L1)"),
                ("logit_contribution", "|logit| contribution of the group"),
                ("logit_change", "old-class logit change due to the group"),
            ],
        ):
            P.plot_degree_diagnostic(runs, key, ax=ax, title=f"{name}: {title}")
        plt.tight_layout()
        plt.show()

**Observed.** The 16 inputs give degree groups with 1, 16, 136 and 816 coefficient rows. Setting both mechanisms is rejected with an error. With the vector `[0.1, 0.1, 0.0, 0.1]` the degree-2 group has **0 trainable coefficients and a measured coefficient change of exactly 0.000 in every experience**, and the trainable total per experience drops from `[34, 612, 5814, 7752, 9690]` to `[34, 68, 4998, 6664, 8330]`.

In the shared-rate diagnostics (class-balanced, rate 3) the raw gradient and the actual coefficient change grow with degree (degree 3 about 600 total change vs about 45 for degree 1 in experiences 3–5) while the logit contribution of the groups is comparable or smaller (about 0.6–0.9 for degree 3, 1.4–3 for degree 1). Note that the coefficient change is a *sum over a group*, and groups differ in size by a factor 816.

*Reduced study (selection seed 0, test seeds 1–2, `ordinary` only).* The study first selected a shared scalar rate of 30 (shuffled) and 3 (class-balanced) on validation data, which already gives 0.601 ± 0.079 and 0.770 ± 0.005 (default rate 0.1: 0.246 and 0.675). The selected vectors do not improve on these: shuffled `d3x2` (degree 3 doubled) gives 0.601 ± 0.106 with forgetting 0.368 vs 0.344; class-balanced `combined` gives 0.774 ± 0.003 with forgetting 0.104 vs 0.097 and newest accuracy 0.728 vs 0.701.

**Interpretation.** A zero rate really freezes its group, and the group no longer counts as trainable. In this reduced run the effect of any per-degree vector is small compared with the effect of the shared scalar rate, and is not consistently better on both accuracy and forgetting: for the class-balanced cell it moves along the trade-off (+0.004 accuracy, +0.007 forgetting, +0.027 newest accuracy). This is **inconclusive**, not negative: the selection used one seed, the test used two, and only `ordinary` was run.

**Caveat.** The full study (all strategies, more seeds) is `experiments/per_degree_lr.py`; its results are deliberately not part of this notebook.

## I.7 · Stability on Split MNIST

**Question.** How variable are the results across seeds on the real dataset, and which of the comparisons above are larger than that variability?

**Why it matters.** Differences between means are only informative if they exceed the spread between seeds. Sections I.3–I.5 are already Split MNIST runs; this section lays out every seed so the spread can be judged, and states the protocol again in one place. Synthetic-ring results (I.1) are a protocol check and say nothing about real data.

**Experiment.** Per-seed final accuracy, newest-experience accuracy and forgetting for the three conditions (I.3) and the six strategies under the default sampling (I.4/I.5).

In [ ]:
if HAVE_MNIST:
    print(
        "protocol:",
        {
            **CIL,
            "seeds": SEEDS,
            "experiences": len(TASK_ORDER),
            "task_order": TASK_ORDER,
            "projection_seed": 0,
        },
    )
    print("\n== expansion conditions, per seed")
    print(metrics.format_table(per_seed_rows(by_condition, key="condition")))
    print("\n== update strategies (shuffled), per seed")
    print(metrics.format_table(per_seed_rows(by_strategy, key="strategy")))
    spread = {
        s: np.std([r["final_mean_accuracy"] for r in runs], ddof=1)
        for s, runs in {**by_condition, **by_strategy}.items()
    }
    print(
        "\nsample std of final mean accuracy across seeds:",
        {k: round(float(v), 3) for k, v in spread.items()},
    )

**Observed.** Across the five seeds the final mean accuracy has a sample standard deviation of at most 0.002 for every condition and strategy except Strategy B (0.009; per seed 0.529–0.553 and 0.529–0.554). The differences discussed above are larger than that: expansion +0.070, `frozen_old` +0.028, class-balanced sampling +0.43, Strategy B about +0.29 at default rates.

**Interpretation.** The ordering of the large effects does not depend on the seed. The small ones (`expanding` vs `fixed_capacity`, 0.010; `ordinary` vs `small_old_lr`, 0.014) exceed the seed spread but not necessarily the spread over projections or data splits, which these runs do not vary.

**Scope.** Sections I.3–I.6 are Split MNIST runs (real data); the synthetic ring of I.1 is only a protocol check. CIFAR-100 was not run, and nothing here is claimed for it.

## I.8 · Research summary

**RQ2 — Does expansion preserve previous knowledge?** At the moment of expansion, yes: scores and predictions are unchanged to $10^{-14}$ for the binary model in I.2 (the ten-degree walk in II.1 reaches $10^{-12}$) and exactly for the ten-class logits (I.2). After training they change, mostly outside the data. Compared with a fixed linear model expansion helps on Split MNIST (0.246 vs 0.176) and equals a fixed-capacity model of the same final size (0.256); it is not shown that gradual growth matters.

**RQ3 — What causes forgetting?** The measured failure is that new-class logits win on old-class inputs (takeover 0.96), not that old logits flip among old classes. Replay with 200 examples per class has a small effect under plain shuffling (+0.054); the sampling policy has a large effect (+0.43 accuracy) at the cost of newest-experience accuracy, and it repeats stored examples about 20 times, so balance and repetition are not separated. Accuracy alone would have hidden this.

**RQ4 — How should old and new dimensions be updated?** Damping or freezing old coefficients reduces flips but not takeover; Strategy B's null-space constraint reduces takeover and trades away newest accuracy, controlled by the reference count. At the default rate and budget none of these is shown to be better than a tuned baseline.

**RQ5 — Does per-degree rate control help?** A zero rate freezes its group exactly. In the reduced study no vector clearly beat the shared rate; the result is inconclusive, and the full study is left to the script.

**RQ6 — Does it transfer to real data?** I.3–I.7 are Split MNIST runs. The ring is a protocol check only; CIFAR-100 was not evaluated.

**Limitations.** Default rate and 3 epochs are far from convergence; one fixed projection; seeds do not vary the data; per-degree study reduced.

# Part II · Sequential binary learning under spatial distribution shift (not CIL)

The label set is fixed (two classes). What changes between experiences is *where in the input space* the samples come from (a vertical strip of the plane), so each experience reveals a different part of the nonlinear boundary $y=1 \iff x_2 > 0.5\sin(3x_1)$. This isolates nonlinear representation and polynomial capacity under distribution shift; Part I reuses its expansion invariant and its update-rule vocabulary. **Forgetting here is the loss of accuracy on earlier *regions*, not on earlier *classes*.**

**Protocol.** Experiences are processed one at a time. At experience *t* the learner may use only the current experience's training samples, its replay buffer (training samples of earlier experiences) and its parameters; evaluation samples are never used for learning or replay. By default the degree grows by one per experience (1, …, 10). The expansion invariant is verified before any training: old coefficients are copied, new ones start at zero, and the scores and probabilities are unchanged on training, evaluation, random and out-of-region inputs.

In [ ]:
from polynomial_cl.binary import (
    STRATEGIES as BINARY_STRATEGIES,
)
from polynomial_cl.binary import (
    config_with_defaults,
    run_continual,
    run_seeds,
    seed_summary_rows,
    step_rows,
    summary_table,
)

BSEED = 0
N_TRAIN, N_TEST = 200, 500
BIN = {
    "lr": 0.5,
    "epochs": 300,
    "m_current": 100,
    "m_replay": 100,
    "capacity": 100,
    "replace": False,
}
BIN_DATA = {"n_train": N_TRAIN, "n_test": N_TEST, "noise": 0.0}
BSEEDS = [0, 1, 2, 3, 4]
bexps = make_experiences(seed=BSEED, n_train=N_TRAIN, n_test=N_TEST)
bresult = run_continual(bexps, config_with_defaults("ordinary", BIN, seed=BSEED))

fig, axs = plt.subplots(1, 2, figsize=(9, 4.2))
S.plot_ground_truth(axs[0], title="Ground truth")
S.plot_ground_truth(axs[1], title="Experiences = vertical strips (training samples)")
for e in bexps:
    S.plot_samples(axs[1], e.train.x, e.train.y, s=6)
    axs[1].axvline(e.window[0], color="k", lw=0.3)
plt.tight_layout()
plt.show()

## II.1 · Expansion walk-through (strategy `ordinary`, seed 0)

Experience 1 trains the initial model. At experience 2 the model is expanded and the invariant is verified *before* training; the second panel of each figure (right after expansion, no training) must equal the first.

In [ ]:
r1, r2 = bresult.records[:2]
print("experience 1: degree", r1.degree_after, "| features:", r1.feature_names)
print(
    f"  training loss {r1.loss_history[0]:.4f} -> {r1.loss_history[-1]:.4f}; accuracy on experience 1: {r1.acc_current:.3f}"
)
S.plot_step(r1, bexps)
plt.show()
print(
    f"experience 2: expansion degree {r2.degree_before} -> {r2.degree_after}: "
    f"max|score diff| = {r2.invariant_score_diff:.2e}, max|proba diff| = {r2.invariant_proba_diff:.2e}"
)
print("  replay batch composition:", r2.batch_counts)
print(S.format_coefficient_table(r2))
S.plot_step(r2, bexps)
plt.show()
S.plot_coefficients(r2)
plt.show()
for t in (3, 6, 10):
    r = bresult.records[t - 1]
    print(
        f"experience {t}: degree {r.degree_before} -> {r.degree_after}, invariant score diff {r.invariant_score_diff:.1e}"
    )
    S.plot_step(r, bexps)
    plt.show()

## II.2 · Learning history and summary

In [ ]:
S.plot_learning_history(bresult)
plt.show()
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
S.plot_accuracy_matrix(bresult, axs[0])
last = bresult.records[-1].model_after
x_all = np.concatenate([e.test.x for e in bexps])
y_all = np.concatenate([e.test.y for e in bexps])
S.plot_correctness(
    last,
    axs[1],
    samples=(x_all, y_all),
    title="Final model: correct/incorrect regions + evaluation samples",
)
plt.tight_layout()
plt.show()
print(metrics.format_table(summary_table(bresult)))

## II.3 · Strategy comparison over five seeds

Same data, evaluation sets, learning rate, epochs and replay budget for every strategy. A preset keeps its defining parameters (for example no replay for `ordinary_no_replay`). Mean ± sample std (ddof = 1) over seeds.

| name | what changes |
|---|---|
| `ordinary_no_replay` | one global learning rate, no replay |
| `ordinary` | one global learning rate, replay |
| `frozen_old` | existing coefficients frozen; only new coefficients train |
| `small_old_lr` | existing coefficients use 0.1 × the learning rate |
| `strategy_b_all` | $f = f_{old} + g$ with $g(x_{ref}) = 0$ on reference inputs (3 buffered inputs per old experience); $g$ over all coefficients |
| `strategy_b_new` | as above, $g$ only over the new coefficients |

Preserving $f_{old}$ at the reference inputs is a constraint at finitely many points; it does not guarantee anything elsewhere.

In [ ]:
bruns = {
    n: run_seeds(BSEEDS, config_with_defaults(n, BIN), BIN_DATA)
    for n in BINARY_STRATEGIES
}
print(metrics.format_table(seed_summary_rows(bruns, key="strategy")))
S.plot_sequential_curves(bruns)
plt.tight_layout()
plt.show()

### Strategy B inspection (seed 0)

`ref_score_change` (change at the reference inputs) should be ~1e-15 by construction; `old_score_change` is measured on the whole old evaluation sets, which are not the reference inputs. `null_dim` is meaningful only for the constrained update; `n/a` means no constraint was applied, not that nothing was learned.

In [ ]:
for n in ("strategy_b_all", "strategy_b_new"):
    print("==", n)
    print(metrics.format_table(step_rows(bruns[n][0]), "{:.3g}"))
x_eval = np.concatenate([e.test.x for e in bexps])
y_eval = np.concatenate([e.test.y for e in bexps])
labels = {
    "ordinary": "ordinary",
    "frozen_old": "frozen_old",
    "strategy_b_all": "Strategy B: all",
    "strategy_b_new": "Strategy B: new",
}
fig, axs = plt.subplots(1, len(labels), figsize=(20, 4.5))
for ax, (name, label) in zip(axs, labels.items()):
    S.plot_correctness(
        bruns[name][0].records[-1].model_after,
        ax=ax,
        samples=(x_eval, y_eval),
        title=label,
        legend=False,
    )
plt.tight_layout()
plt.show()

## II.4 · Does the degree schedule matter?

Three conditions on the same experiences (paired by seed) with identical replay, epochs and optimiser: degree 1 throughout (no expansion), growing 1, …, 10, and degree 10 from experience 1 (same final capacity, no gradual growth). *Growing vs degree 1* asks whether added capacity helps at all; *growing vs degree 10* asks whether adding it gradually matters. The same experiment is `experiments/synthetic_degree_ablation.py`.

In [ ]:
ablation = {
    label: run_seeds(
        BSEEDS, config_with_defaults("ordinary", BIN, degrees=degrees), BIN_DATA
    )
    for label, degrees in {
        "fixed degree 1 (no expansion)": [1] * 10,
        "grow 1..10": None,
        "fixed degree 10 (capacity control)": [10] * 10,
    }.items()
}
print(metrics.format_table(seed_summary_rows(ablation, key="condition")))
S.plot_sequential_curves(ablation)
plt.tight_layout()
plt.show()

## II.5 · Why does Strategy B fit the curve worse? Reference points versus optimisation

Strategy B pins $g(x_{ref}) = 0$ at 3 buffered inputs per old experience. Two explanations are compared on five seeds: *too little optimisation* (10× epochs) versus *the constraint itself* (fewer or more reference points per old experience). `mean_old_score_change` measures how much the old function actually moves on the old evaluation sets.

In [ ]:
variants = {
    "ordinary": config_with_defaults("ordinary", BIN),
    "strategy_b_all, 1 ref/exp": config_with_defaults(
        "strategy_b_all", BIN, n_ref_per_exp=1
    ),
    "strategy_b_all, 2 ref/exp": config_with_defaults(
        "strategy_b_all", BIN, n_ref_per_exp=2
    ),
    "strategy_b_all, 3 ref/exp (default)": config_with_defaults("strategy_b_all", BIN),
    "strategy_b_all, 5 ref/exp": config_with_defaults(
        "strategy_b_all", BIN, n_ref_per_exp=5
    ),
    "strategy_b_all, 3 ref/exp, 10x epochs": config_with_defaults(
        "strategy_b_all", {**BIN, "epochs": 10 * BIN["epochs"]}
    ),
    "frozen_old": config_with_defaults("frozen_old", BIN),
}
print(
    metrics.format_table(
        seed_summary_rows(
            {k: run_seeds(BSEEDS, c, BIN_DATA) for k, c in variants.items()}
        )
    )
)

## II.6 · Per-coefficient learning rates (`lr_feature_power`)

`lr_old_factor` gives old and new coefficients different rates; `lr_feature_power = p` additionally gives coefficient *i* the rate `lr * (max_rms / rms_i) ** p`, so small high-degree features take larger steps (p = 1 is diagonal preconditioning). 300 epochs with a shared rate are not converged here, so the rate allocation matters. The comparison repeats the expansion question with both settings.

In [ ]:
setups = {
    "shared rate (p=0, lr=0.5)": {"lr": 0.5, "lr_feature_power": 0.0},
    "per-feature rate (p=1, lr=4)": {"lr": 4.0, "lr_feature_power": 1.0},
}
degree_sets = {
    "degree 1 (no expansion)": [1] * 10,
    "grow 1..10": None,
    "fixed degree 10": [10] * 10,
}
table = {}
for setup, kw in setups.items():
    for cond, degrees in degree_sets.items():
        table[f"{cond} | {setup}"] = run_seeds(
            BSEEDS,
            config_with_defaults("ordinary", {**BIN, **kw}, degrees=degrees),
            BIN_DATA,
        )
print(metrics.format_table(seed_summary_rows(table, key="condition | setup")))

## Part II summary (binary, not CIL)

**Observed (rate 0.5, 300 epochs, 10 experiences, five seeds; mean ± sample std).**

* *Strategies.* Final mean accuracy: `ordinary` 0.965 ± 0.004 (forgetting 0.008), `small_old_lr` 0.916, `ordinary_no_replay` 0.866 (0.125), `frozen_old` 0.842, `strategy_b_new` 0.844, `strategy_b_all` 0.840 ± 0.018 (0.000). The Strategy B reference score change is at most 3e-15 in every experience, so the constraint holds to rounding at the reference inputs; for `strategy_b_new` the null dimension is 0 and the old function does not move at all.
* *Degree schedule.* Fixed degree 1: 0.910 (forgetting 0.043); growing 1…10: 0.965 (0.008); fixed degree 10: 0.970 (0.011).
* *Reference points.* Strategy B with 1, 2, 3, 5 references per old experience: 0.912, 0.848, 0.840, 0.847, with mean old-score change 1.99, 0.90, 0.30, 0.01; with 10× the epochs 0.839 ± 0.031 (old-score change 2.64).
* *Per-coefficient rates* (`lr_feature_power = 1`, `lr = 4`): growing 0.992 ± 0.002 (forgetting 0.002), fixed degree 10 0.991, fixed degree 1 0.910.

**Interpretation.** Added capacity helps clearly here (0.910 → 0.965); adding it gradually or all at once is not distinguishable (0.965 vs 0.970, about one standard deviation). Strategy B fits the sine curve worse than `ordinary` because of the constraint, not because of too little optimisation: 10× the epochs does not change its accuracy, and the number of reference points moves it along a preservation/fit trade-off. A constraint at three reference points per experience protects those points only: with 10× the epochs the old function still moves substantially elsewhere (2.64). The per-feature-rate setting improves the expansion conditions, but it was fixed beforehand rather than selected here, so it shows that rate allocation matters, not that this setting is optimal.

**Scope.** This is spatial distribution shift with two classes; forgetting here is the loss of accuracy on earlier *regions* and does not transfer as a claim to class-incremental learning.